# TC-WPN — Phase 5: auxiliary dose-response and K-shot sensitivity

Supervisor's Steps 2 and 3. **Every model-selection number in this notebook comes
from the VALIDATION split. The test set is locked** — it is touched once, at the
end, for the single finally-chosen configuration.

## Section 0 is free and settles a question before any GPU time

Two corrections to the Phase 4 reading come out of arithmetic alone, and both
change what the next experiments should be.

### The temporal correlation of −0.99 is not evidence of learning

`temporal_aux` reported `corr(weight, days_before_index) = −0.992`. But

    w_i = exp(-lambda * days_i / 365)

is a strictly decreasing function of `days` for **any** lambda > 0, so the
correlation is near −1 by construction. Simulating it confirms this:

| lambda | mean corr(w, days) |
|---|---:|
| 0.001 | −1.00000 |
| 0.05 | −0.99997 |
| 0.4198 (learned) | −0.99783 |
| 2.0 | −0.95615 |
| 10.0 | −0.76224 |

Even lambda = 0.001, which is effectively no weighting at all, gives −1.000. So
this number is a **correctness check** that days are wired to weights — valuable,
but it does not show the mechanism learned a temporal preference. Do not write
that in the paper; a reviewer who differentiates the formula will catch it.

The quantity that *would* show learning is lambda's movement, and it went the
wrong way for the hypothesis: 0.4898 → 0.4198 in `temporal_aux` and 0.4884 →
0.4378 in `tcwpn_full`. Both moved **toward zero**, i.e. toward flatter, more
uniform weighting. Gradient descent was reducing the mechanism's influence.

### At K = 1 the mechanisms are mathematically inert

Both weights are normalised across the support notes of a class. With one note
the normalised weight is 1.0 regardless of lambda or beta. So `aux_only`,
`temporal_aux` and `tcwpn_full` **must** coincide at K = 1.

That rules out the "helps most under extreme support scarcity" hypothesis in its
K=1 form. K = 1 is still worth running, but as a **null control**: if the three
configurations differ there by more than seed noise, something is wired wrong.
The mechanism can only act at K >= 2, so the interesting comparison is K = 3, 5, 10.

### What the weights actually cost

Effective support size `K_eff = K^H_norm`, from your Phase 4 entropies:

| config | H_norm | K_eff of 5 |
|---|---:|---:|
| aux_only | 1.00000 | 5.00 |
| pcw_aux | 0.99813 | 4.98 |
| temporal_aux | 0.94679 | 4.59 |
| tcwpn_full | 0.94323 | 4.56 |
| tcwpn_full, worst 5% | 0.79531 | 3.60 |

The temporal weighting spends about **0.7 of a support example** on average, and
1.4 in the worst 5% of episodes. That is the mechanism's price. For it to break
even, the recency information has to be worth more than the discarded note — and
at K = 5 it evidently is not. This predicts the effect should be *less* harmful
at K = 10, where a note is a smaller fraction of the support set.

In [ ]:
!rm -rf /kaggle/working/tcwpn_test
!git clone -q https://github.com/dulhara79/tcwpn_test.git /kaggle/working/tcwpn_test
%cd /kaggle/working/tcwpn_test
!pip install -q -r requirements.txt 2>&1 | tail -2
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"]="1"; os.environ["TRANSFORMERS_VERBOSITY"]="error"
os.environ["TOKENIZERS_PARALLELISM"]="false"; os.environ["PYTHONPATH"]="src"

import subprocess, sys
r=subprocess.run([sys.executable,"-m","pytest","tests/test_repo_layout.py",
                  "tests/test_call_arity.py","-q","--no-header"],
                 capture_output=True,text=True,env=os.environ)
print(r.stdout[-1200:])
if r.returncode!=0: raise SystemExit("repository layout broken")

In [ ]:
from pathlib import Path
import glob, os, shutil

STAGE_A = next((c.parent for c in Path("/kaggle/input").rglob("plans")
                if (c.parent/"pkl").exists()), None)
if STAGE_A is None: raise SystemExit("Stage A dataset not found")
PKL_DIR, PLAN_DIR = "/kaggle/working/pkl", str(STAGE_A/"plans")
STEM, RESULTS, LOGS = "psych_mimic4idx", "/kaggle/working/results", "/kaggle/working/logs"
!mkdir -p {PKL_DIR} {LOGS}
!cp {STAGE_A}/pkl/*.pkl {PKL_DIR}/
for src in glob.glob("/kaggle/input/**/*_blind-*.pkl", recursive=True):
    d=os.path.join(PKL_DIR,os.path.basename(src))
    if not os.path.exists(d): shutil.copy2(src,d)
print("plans available:", sorted(os.path.basename(f) for f in glob.glob(f"{PLAN_DIR}/*_k*.json"))[:8])

## Section 1 — counterfactual: what do the weights actually do?

Free of training cost, and it fixes a confound in the Phase 4 table.

`analyse_mechanisms.py --reference` reported prototype cosine 0.567 between
`tcwpn_full` and `aux_only`. Those are **two separately trained models**, so that
number mixes the encoder having converged differently with the weighting being
non-uniform. Only the second is the mechanism under test, and two independently
trained BERT encoders differ by that much routinely.

The clean measurement holds everything fixed except the weights: inside one
model, on one set of embeddings, build the prototype with the learned weights and
again with uniform weights. The bottom line is `decision_flip_rate` — the
fraction of query notes whose predicted class changes when the weighting is
switched off. If that is near zero, the mechanism cannot move AUROC.

Uses the seed-42 checkpoints, validation split.

In [ ]:
CONFIGS_CF = ["aux_only", "temporal_aux", "pcw_aux", "tcwpn_full"]
K, SEED = 5, 42
found = {}
for cfg in CONFIGS_CF:
    name=f"{cfg}_k{K}_seed{SEED}"
    hits=glob.glob(f"/kaggle/input/**/{name}/best.pt", recursive=True)
    if not hits:
        print(f"no checkpoint for {cfg} -- add its Phase 3B notebook output as an input")
        continue
    src=os.path.dirname(sorted(hits)[0]); dst=f"{RESULTS}/{STEM}/{name}"
    os.makedirs(dst,exist_ok=True)
    for f in os.listdir(src):
        q=os.path.join(src,f)
        if os.path.isfile(q): shutil.copy2(q,dst)
    found[cfg]=dst
print("available:", list(found))

for cfg, run in found.items():
    print("="*72)
    !python -m scripts.counterfactual_prototype --run {run} \
        --pkl-dir {PKL_DIR} --plan-dir {PLAN_DIR} \
        --split val --episodes 300 \
        --out /kaggle/working/counterfactual_{cfg}.json

In [ ]:
import json, pandas as pd
rows=[]
for f in sorted(glob.glob("/kaggle/working/counterfactual_*.json")):
    d=json.load(open(f)); g=lambda k: round(d[k]["mean"],5) if d.get(k) else None
    rows.append({"config":d["preset"],
                 "K_eff":g("effective_K"),
                 "proto_cos_w_vs_uniform":g("proto_cos_weighted_vs_uniform"),
                 "decision_flip_rate":g("decision_flip_rate"),
                 "score_shift":g("score_shift_abs"),
                 "AUROC_weights_on":round(d["episode_auroc_weighted"],5),
                 "AUROC_weights_off":round(d["episode_auroc_uniform"],5),
                 "delta":round(d["episode_auroc_delta"],5)})
if rows:
    t=pd.DataFrame(rows).set_index("config")
    print(t.to_string()); t.to_csv("/kaggle/working/phase5_counterfactual.csv")
    print()
    print("AUROC_weights_off is THIS model with its weighting disabled at inference.")
    print("If delta is ~0, the mechanism is not contributing at K=5, and the")
    print("comparison is internal so no encoder difference can explain it away.")

## Section 2 — auxiliary dose-response (supervisor Step 2)

Phase 3B showed the jump from ~0.50 to ~0.74 is the auxiliary CE head, not the
weightings. This measures its dose-response with both mechanisms **off**, so
nothing else varies.

Two endpoints already exist: weight 0.0 is `protonet_temp` (val ≈ chance) and
weight 0.3 is `aux_only`. This adds 0.1, 0.25, 0.5, 1.0, 2.0 — five runs, one
seed, roughly four hours.

**Validation only.** Do not evaluate these on test. Picking a weight by test
AUROC would convert the test set into a tuning set and forfeit the credibility
the leakage certificate bought.

In [ ]:
import time, subprocess
def run(cmd, logfile, tail=20):
    t0=time.time()
    with open(logfile,"w") as fh:
        p=subprocess.run(cmd,stdout=fh,stderr=subprocess.STDOUT,env=os.environ)
    print("\n".join(open(logfile).read().splitlines()[-tail:]))
    print(f"[exit {p.returncode} | {(time.time()-t0)/60:.1f} min]")
    return p.returncode

AUX_SWEEP = ["aux_w0.1", "aux_w0.25", "aux_w0.5", "aux_w1.0", "aux_w2.0"]
STOP_AFTER_H = 9.0
t_start = time.time()

for cfg in AUX_SWEEP:
    if (time.time()-t_start)/3600 > STOP_AFTER_H:
        print(f"stopping to stay under the session cap; remaining: {AUX_SWEEP[AUX_SWEEP.index(cfg):]}")
        break
    run_dir=f"{RESULTS}/{STEM}/{cfg}_k{K}_seed{SEED}"
    if os.path.exists(f"{run_dir}/eval_val.json"):
        print(f"skip {cfg}"); continue
    print("="*72); print(f"TRAIN {cfg}"); print("="*72)
    rc=run(["python","-m","scripts.train","--config",f"configs/{cfg}.yaml",
            "--k",str(K),"--seed",str(SEED),"--stem",STEM,
            "--pkl-dir",PKL_DIR,"--plan-dir",PLAN_DIR,"--results",RESULTS],
           f"{LOGS}/train_{cfg}.log")
    if rc!=0: continue
    run(["python","-m","scripts.evaluate","--run",run_dir,"--split","val",
         "--pkl-dir",PKL_DIR,"--plan-dir",PLAN_DIR,"--bootstrap","2000"],
        f"{LOGS}/eval_val_{cfg}.log")
    if os.path.exists(f"{run_dir}/best.pt"):
        os.remove(f"{run_dir}/best.pt")
        print("[removed checkpoint]")

In [ ]:
import json, glob, pandas as pd
rows=[]
for cfg in ["protonet_temp"]+["aux_w0.1","aux_w0.25"]+["aux_only"]+["aux_w0.5","aux_w1.0","aux_w2.0"]:
    f=f"{RESULTS}/{STEM}/{cfg}_k{K}_seed{SEED}/eval_val.json"
    if not os.path.exists(f): continue
    m=json.load(open(f))["metrics"]
    w={"protonet_temp":0.0,"aux_w0.1":0.1,"aux_w0.25":0.25,"aux_only":0.3,
       "aux_w0.5":0.5,"aux_w1.0":1.0,"aux_w2.0":2.0}[cfg]
    rows.append({"config":cfg,"aux_weight":w,"val_AUROC":round(m["auroc"],4),
                 "val_PR_AUC":round(m["pr_auc"],4)})
if rows:
    t=pd.DataFrame(rows).sort_values("aux_weight")
    print(t.to_string(index=False)); t.to_csv("/kaggle/working/phase5_aux_sweep.csv",index=False)
    print()
    print("A rising curve that plateaus means the auxiliary objective is the")
    print("active ingredient and 0.3 was already near the plateau. A peak")
    print("elsewhere is a validation-selected improvement you may then confirm")
    print("ONCE on test, with the choice pre-registered in the paper.")

## Section 3 — K-shot sensitivity (supervisor Step 3)

Three configurations × K ∈ {1, 3, 5, 10}. K = 5 is already done, so this is nine
new runs — about seven hours, so expect two sessions.

Read K = 1 as a **null control**: the three configurations must agree there,
because normalised weights over a single support note are 1.0 by definition. If
they disagree by more than seed noise, stop and check the wiring.

The real question is the K = 10 column. If the weighting's cost is that it
discards support examples, that cost should shrink as K grows.

In [ ]:
K_CONFIGS = ["aux_only", "temporal_aux", "tcwpn_full"]
K_VALUES  = [1, 3, 10]          # 5 already done in Phase 3B
t_start = time.time()

for KK in K_VALUES:
    if not os.path.exists(f"{PLAN_DIR}/{STEM}_val_k{KK}.json"):
        print(f"no plan for K={KK}; skipping"); continue
    for cfg in K_CONFIGS:
        if (time.time()-t_start)/3600 > STOP_AFTER_H:
            print("stopping to stay under the session cap"); break
        run_dir=f"{RESULTS}/{STEM}/{cfg}_k{KK}_seed{SEED}"
        if os.path.exists(f"{run_dir}/eval_val.json"):
            print(f"skip {cfg} K={KK}"); continue
        print("="*72); print(f"TRAIN {cfg}  K={KK}"); print("="*72)
        rc=run(["python","-m","scripts.train","--config",f"configs/{cfg}.yaml",
                "--k",str(KK),"--seed",str(SEED),"--stem",STEM,
                "--pkl-dir",PKL_DIR,"--plan-dir",PLAN_DIR,"--results",RESULTS],
               f"{LOGS}/train_{cfg}_k{KK}.log")
        if rc!=0: continue
        run(["python","-m","scripts.evaluate","--run",run_dir,"--split","val",
             "--pkl-dir",PKL_DIR,"--plan-dir",PLAN_DIR,"--bootstrap","2000"],
            f"{LOGS}/eval_val_{cfg}_k{KK}.log")
        if os.path.exists(f"{run_dir}/best.pt"): os.remove(f"{run_dir}/best.pt")

In [ ]:
import json, glob, pandas as pd
rows=[]
for cfg in K_CONFIGS:
    for KK in [1,3,5,10]:
        f=f"{RESULTS}/{STEM}/{cfg}_k{KK}_seed{SEED}/eval_val.json"
        if not os.path.exists(f): continue
        rows.append({"config":cfg,"K":KK,
                     "val_AUROC":round(json.load(open(f))["metrics"]["auroc"],4)})
if rows:
    t=pd.DataFrame(rows).pivot(index="config",columns="K",values="val_AUROC")
    print(t.to_string()); t.to_csv("/kaggle/working/phase5_kshot.csv")
    print()
    if 1 in t.columns:
        spread=t[1].max()-t[1].min()
        print(f"K=1 spread across configurations: {spread:.4f}")
        print("Expected ~0: the mechanisms cannot act on a single support note.")
        print("A large spread means something is wired wrong -- investigate before")
        print("interpreting any other column.")

## What this can and cannot conclude

**Do not** write that TC-WPN improves few-shot anxiety detection. One seed in
five, median Δ negative, paired p = 0.886, and the sign flips when seed 43 is
dropped.

**Do** treat the Phase 3B numbers as the locked primary benchmark:

> aux_only 0.7371 ± 0.0081, TC-WPN 0.7377 ± 0.0031, paired Δ = +0.0006,
> 95% CI [−0.0103, +0.0115], p = 0.886.

The honest framing, which your supervisor set out in their section 22, Outcome B:

> Under auxiliary-controlled ablation across five seeds on a patient-disjoint,
> ICD-labelled cohort with a zero-leakage episode certificate, neither
> index-relative temporal weighting nor prototype-consistency weighting improved
> discrimination over the auxiliary-controlled prototypical baseline. Diagnostic
> analysis shows the temporal mechanism reduces the effective support size from
> 5.00 to 4.59 examples, and the auxiliary supervised objective accounts for
> essentially all of the improvement over standard prototypical training
> (0.50 → 0.74).

That is a complete, defensible contribution: a leakage-controlled benchmark, a
controlled ablation, and a mechanistic account of a negative result.

**One caveat on the blinding table.** Three configurations have blinded
evaluations but `aux_only` — the control — does not:

| config | original | anxiety-blind | margin retained |
|---|---:|---:|---:|
| aux_only | 0.7432 | *missing* | — |
| temporal_aux | 0.7362 | 0.6351 | 0.572 |
| pcw_aux | 0.7208 | 0.6162 | 0.526 |
| tcwpn_full | 0.7379 | 0.6284 | 0.540 |

Without the control you cannot say TC-WPN is more or less shortcut-reliant than
the baseline. It is inference-only on a checkpoint you already have — a few
minutes, and it completes the table.

In [ ]:
# Fill the missing control cell in the blinding table. Test split, because the
# other three blinded numbers are on test and must be comparable.
run_dir = found.get("aux_only")
if run_dir:
    for level in ["anxiety","anx_meds"]:
        if os.path.exists(f"{run_dir}/eval_test_blind-{level}.json"): continue
        if not os.path.exists(f"{PKL_DIR}/{STEM}_test_blind-{level}.pkl"):
            print(f"missing blinded pkl for {level} -- run Stage A2 first"); continue
        run(["python","-m","scripts.evaluate","--run",run_dir,"--split","test",
             "--blind",level,"--pkl-dir",PKL_DIR,"--plan-dir",PLAN_DIR,
             "--bootstrap","2000"], f"{LOGS}/eval_blind_{level}_aux_only.log")
else:
    print("aux_only checkpoint not available in this session")